OPZETTEN TABEL


In [38]:
#imports
from pyspark.sql.functions import *
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, IntegerType, ByteType
from datetime import datetime
import ConnectionConfig as cc

In [39]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_TREASURE_TYPE",4)
spark.getActiveSession()

Environment variables are set...


In [40]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")
cc.config.sections()

['default', 'tutorial_op', 'catchem', 'kafka']

In [41]:
#get info
#treasure
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure")

#stage
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_stages") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("stages")

In [42]:
#select maken voor info voor de tabel
df_dim_treasure_type = spark.sql("""
    WITH stages_count AS (
        SELECT
            treasure_id,
            COUNT(DISTINCT stages_id) as size
        FROM stages
        GROUP BY treasure_id
    ),
    treasure_with_size AS (
        SELECT
            t.difficulty,
            t.terrain,
            COALESCE(s.size, 0) as size
        FROM treasure t
        LEFT JOIN stages_count s ON t.id = s.treasure_id
    )
    SELECT DISTINCT
        difficulty as Difficulty,
        terrain as Terrain,
        size as Size
    FROM treasure_with_size
    ORDER BY difficulty, terrain, size
""")

In [43]:
#deltatabel maken
spark.sql("DROP TABLE IF EXISTS dimUser")

DeltaTable.createOrReplace(spark) \
    .tableName("dimTreasureType") \
    .addColumn("TreasureTypeSurKey", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("Difficulty", IntegerType()) \
    .addColumn("Terrain", IntegerType()) \
    .addColumn("Size", LongType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

In [44]:
df_dim_treasure_type.write.format("delta").mode("overwrite").saveAsTable("dimTreasureType")
spark.sql("SELECT * FROM dimTreasureType").show(100)

+------------------+----------+-------+----+
|TreasureTypeSurKey|Difficulty|Terrain|Size|
+------------------+----------+-------+----+
|                 0|         0|      0|   1|
|                 1|         0|      0|   2|
|                 2|         0|      0|   3|
|                 3|         0|      0|   4|
|                 4|         0|      0|   5|
|                 5|         0|      0|   6|
|                 6|         0|      0|   7|
|                 7|         0|      0|   8|
|                 8|         0|      0|   9|
|                 9|         0|      0|  10|
|                10|         0|      1|   1|
|                11|         0|      1|   2|
|                12|         0|      1|   3|
|                13|         0|      1|   4|
|                14|         0|      1|   5|
|                15|         0|      1|   6|
|                16|         0|      1|   7|
|                17|         0|      1|   8|
|                18|         0|      1|   9|
|         

In [45]:
spark.stop()